In [ ]:
!pip install -U bitsandbytes>=0.46.1


In [ ]:
!pip install -U torchao

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

CSV_PATH = "/content/drive/MyDrive/parallel_1000_dostoyevsky_splited.csv"
NEUTRAL_COL = "neutral_text"
STYLED_COL = "styled_text"

OUTPUT_DIR = "./lora_style_adapter"
SYSTEM_PROMPT = (
    "Ты — стилист текста. Перепиши данный нейтральный текст в стиле автора, "
    "сохранив смысл, но изменив манеру изложения, лексику и интонацию."
)

MAX_SEQ_LEN = 512
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

In [ ]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token_id is None or tokenizer.pad_token_id == tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.unk_token if tokenizer.unk_token else "<pad>"
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

tokenizer.padding_side = "right"

In [ ]:

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:

def to_chat_format(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example[NEUTRAL_COL]},
        {"role": "assistant", "content": example[STYLED_COL]},
    ]
    return {"messages": messages}


In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
dataset = Dataset.from_pandas(df)

dataset = dataset.map(to_chat_format, remove_columns=dataset.column_names)

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",
    bf16=True,
    packing=False,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()

In [ ]:
trainer.save_model('/content/drive/MyDrive')
tokenizer.save_pretrained('/content/drive/MyDrive')

In [ ]:
model.eval()

def stylize(neutral_text: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": neutral_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )


    new_tokens = output[0][inputs["input_ids"].shape[1]:] # отрезаем промпт
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
stylize('Здравствуйте! Все таки смог обучить нейронную сеть, теперь она пишет правильно.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu",
)
peft_model = PeftModel.from_pretrained(base, "/content/drive/MyDrive")
merged = peft_model.merge_and_unload()

if hasattr(merged.config, "quantization_config"):
    del merged.config.quantization_config

merged.save_pretrained("/content/drive/MyDrive/merged1", safe_serialization=True)

AutoTokenizer.from_pretrained("/content/drive/MyDrive").save_pretrained("/content/drive/MyDrive/merged1")